# Fig. (appendix) — SNR / power sweep

Interactive front-end. Loader/summary helpers come from `build_fig_snr.py`;
**`build_figure` is inlined below as an editable cell** so you can override style
without editing the module.

- **(top)** min-SINR vs SNR; **(bottom)** sum-SCNR vs SNR.
- A sanity check that both metrics scale monotonically with the transmit power
  budget P_max/σ_n² and the algorithm ordering is preserved.
- Dashed line marks the deployed operating point SNR\* (= `channel.snr_db`, 137 dB).

**Note:** `channel.snr_db` is a *transmit-side* P_max/σ_n²; sweep values must bracket
the 137 dB operating point (not the registry's generic [-10..25] default, which would
be far below it).

**Compatible experiment:** `snr_sweep` (`kind=='sweep'`).

In [ ]:
import sys, logging
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

FIG_DIR = Path.cwd()
if str(FIG_DIR) not in sys.path:
    sys.path.insert(0, str(FIG_DIR))

import build_fig_snr as B   # loader + helpers (single source of truth)
from cordis.plotting import apply_paper_style, figsize, plot_sweep, save_figure
from cordis.plotting.style import ALGORITHM_STYLE   # tweak here to restyle

USE_TEX = True   # set False on a node without pdflatex
apply_paper_style()
logging.basicConfig(level=logging.INFO, format='%(levelname)-7s %(message)s')

## 1. Load the SNR-sweep run

`RESULT_DIR = None` auto-picks the newest run (`array_<jobid>_aggregated/` preferred;
`latest` symlink never used).

In [ ]:
RESULT_DIR = None   # e.g. 'results/exp_snr_sweep/array_12345_aggregated'
result, result_dir = B.load_sweep(RESULT_DIR, experiment='snr_sweep')
print('run:', result_dir)
print('SNR grid [dB]:', sorted(result.sweep_results.keys()))

## 2. Resolve algorithms, SCNR metric, operating point

In [ ]:
SNR_STAR    = None                     # None -> auto-detect channel.snr_db from metadata
ONLY        = list(B.PREFERRED)        # Centralized / CORDIS-ADMM / CORDIS-Split
SCNR_METRIC = None                     # None -> first available of B.SCNR_METRIC_PREFERENCE

snr_star, detected = (SNR_STAR, True) if SNR_STAR is not None \
    else B._detect_snr_star(result)
only = B._present_algorithms(result, ONLY)
scnr_metric = SCNR_METRIC or B._select_scnr_metric(result, only=only)
print(f'SNR* = {snr_star:g} dB  (detected={detected})')
print('algorithms:', only)
print('scnr metric:', scnr_metric)

## 3. Numeric summary (caption sanity check)

In [ ]:
summary = B.collect_summary(result, only, scnr_metric, snr_star)
B._print_summary(result, summary, scnr_metric, snr_star)

## 4. `build_figure` — editable copy

The **exact** function from `build_fig_snr.py`, inlined so you can edit it and re-run.
Mutate `ALGORITHM_STYLE` in the setup cell to restyle without editing the body.

In [ ]:
# Bind the module-level names the function body references.
from typing import Optional, Sequence   # the inlined signature uses these
PREFERRED              = B.PREFERRED
SINR_METRIC            = B.SINR_METRIC
SNR_STAR_DEFAULT_DB    = B.SNR_STAR_DEFAULT_DB
_SCNR_YLABEL           = B._SCNR_YLABEL
_present_algorithms    = B._present_algorithms
_select_scnr_metric    = B._select_scnr_metric
_in_range              = B._in_range
logger                 = logging.getLogger('fig_snr.nb')

In [ ]:
def build_figure(result,
                 *,
                 only: Optional[Sequence[str]] = None,
                 scnr_metric: Optional[str] = None,
                 snr_star: float = SNR_STAR_DEFAULT_DB,
                 use_tex: bool = True):
    """Assemble the stacked SNR-sweep figure; return ``(fig, only, scnr_metric)``.

    Built directly on ``cordis.plotting.plot_sweep`` so the paper builder stays
    decoupled from scripts/.  Reuses the project per-algorithm style.
    """
    import matplotlib
    if use_tex is False:
        matplotlib.rcParams["text.usetex"] = False
    import matplotlib.pyplot as plt
    from cordis.plotting import apply_paper_style, figsize, plot_sweep

    apply_paper_style()
    if use_tex is False:
        matplotlib.rcParams["text.usetex"] = False

    only = list(only) if only else _present_algorithms(result, PREFERRED)
    if scnr_metric is None:
        scnr_metric = _select_scnr_metric(result, only=only)

    axis = result.sweep_axis
    xlabel = axis.display or r"SNR $P_{\max}/\sigma_n^2$ [dB]"

    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=figsize(width="single", aspect=3.5 / 2.6),
        sharex=True, gridspec_kw={"hspace": 0.12},
    )

    plot_sweep(result.sweep_results, metric=SINR_METRIC, ax=ax_top,
               xlabel="", ylabel=r"min-SINR [dB]", only=only, log_x=False)
    ax_top.set_title("Scaling with transmit power")

    if scnr_metric is not None:
        plot_sweep(result.sweep_results, metric=scnr_metric, ax=ax_bot,
                   xlabel=xlabel,
                   ylabel=_SCNR_YLABEL.get(scnr_metric, scnr_metric),
                   only=only, log_x=False)
    else:
        logger.warning("No SCNR metric available; bottom panel left empty.")
        ax_bot.set_xlabel(xlabel)

    # operating-point marker SNR* (only if inside the swept range)
    if _in_range(result, snr_star):
        for ax in (ax_top, ax_bot):
            ax.axvline(snr_star, ls="--", lw=0.9, color="0.45", zorder=0)
        ax_top.text(snr_star, 0.04, r"$\mathrm{SNR}^\star$",
                    transform=ax_top.get_xaxis_transform(),
                    ha="right", va="bottom", fontsize="small", color="0.35")
    else:
        logger.warning("snr_star=%.3g dB is outside the swept range; "
                       "operating-point marker skipped.", snr_star)

    if ax_bot.get_legend() is not None:
        ax_bot.get_legend().remove()

    # NB: no fig.tight_layout() — conflicts with the shared-x / hspace stacked
    # layout.  save_figure() uses bbox_inches='tight'.
    return fig, only, scnr_metric


## 5. Render

In [ ]:
fig, used_only, used_scnr_metric = build_figure(
    result, only=only, scnr_metric=scnr_metric,
    snr_star=snr_star, use_tex=USE_TEX,
)
fig

## 6. Save to `paper/figures/`

In [ ]:
out_stem = FIG_DIR.parents[1] / 'figures' / 'fig_snr'
paths = save_figure(
    fig, out_stem, formats=('pdf',),
    metadata={
        'Figure': 'fig_snr',
        'SnrStarDB': f'{snr_star:g}',
        'Algorithms': ', '.join(used_only),
        'ScnrMetric': str(used_scnr_metric),
        'Run': result_dir.name,
    },
)
for p in paths:
    print('wrote', p)